# Instruction-Following Models: From Base LLM to Chat

In the previous notebook we continued pre-training — teaching a model *more language*. But a pre-trained model just predicts the next token. It doesn't follow instructions, answer questions, or have a conversation.

**This notebook explores instruction-tuned (chat) models:**
1. The difference between a **base** model and an **instruct/chat** model
2. How chat templates and special tokens work
3. Hands-on with **SmolLM2** (HuggingFace, 135M–1.7B params) and **Qwen2.5** (Alibaba, 0.5B–72B params)
4. Comparing model outputs and understanding trade-offs

---

In [ ]:
%pip install -q transformers torch accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1 — Base Model vs Instruct Model

A **base model** (e.g. `gpt2`, `Qwen/Qwen2.5-0.5B`) is trained to predict the next token. If you give it a question, it doesn't *answer* — it *continues* the text as if it were a document.

An **instruct model** (e.g. `Qwen/Qwen2.5-0.5B-Instruct`) is the same architecture, further fine-tuned on instruction-response pairs so it actually follows instructions.

Let's see the difference.

In [ ]:
# Load a small base model
base_name = "Qwen/Qwen2.5-0.5B"
base_tokenizer = AutoTokenizer.from_pretrained(base_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_name, torch_dtype=torch.float32
).to(device)
base_model.eval()

print(f"Loaded: {base_name}")
print(f"Parameters: {sum(p.numel() for p in base_model.parameters()):,}")

In [ ]:
def generate_text(prompt, model, tokenizer, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


# Ask the BASE model a question
prompt = "What is the capital of France?"
print(f'Prompt: "{prompt}"\n')
print("--- Base model response ---")
print(generate_text(prompt, base_model, base_tokenizer))
print("\n" + "=" * 70)
print("Notice: it doesn't answer — it continues the text as if writing a quiz or article.")

In [ ]:
# Now load the INSTRUCT version of the same model
instruct_name = "Qwen/Qwen2.5-0.5B-Instruct"
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_name)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_name, torch_dtype=torch.float32
).to(device)
instruct_model.eval()

print(f"Loaded: {instruct_name}")

In [ ]:
# Ask the INSTRUCT model the same question — using the chat template
messages = [{"role": "user", "content": "What is the capital of France?"}]

chat_input = instruct_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print("--- Chat template (what the model actually sees) ---")
print(repr(chat_input))

print("\n--- Instruct model response ---")
inputs = instruct_tokenizer(chat_input, return_tensors="pt").to(device)
with torch.no_grad():
    output = instruct_model.generate(
        **inputs, max_new_tokens=100,
        temperature=0.7, do_sample=True,
        pad_token_id=instruct_tokenizer.eos_token_id,
    )
# Only show the generated part (skip the prompt)
generated = instruct_tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
)
print(generated)

## 2 — Chat Templates: The Secret Ingredient

Instruct models don't magically understand questions. They were trained on text formatted with **special tokens** that mark the boundary between user input and model response.

Each model family has its own chat template. Let's inspect them.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is NLP?"},
    {"role": "assistant", "content": "NLP stands for Natural Language Processing."},
    {"role": "user", "content": "Give me an example."},
]

# Qwen template
qwen_formatted = instruct_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print("=== Qwen Chat Template ===")
print(qwen_formatted)
print()

In [ ]:
# Inspect the special tokens
print("Qwen special tokens:")
for name in ["bos_token", "eos_token", "pad_token"]:
    token = getattr(instruct_tokenizer, name, None)
    print(f"  {name}: {repr(token)}")

print(f"\nAdditional special tokens:")
for tok in instruct_tokenizer.additional_special_tokens[:10]:
    print(f"  {repr(tok)}")

## 3 — SmolLM2: A Tiny but Capable Chat Model

[SmolLM2](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) is a family of small models from HuggingFace (135M, 360M, 1.7B parameters). Small enough to run on any laptop, yet trained to follow instructions.

Let's load the 360M instruct variant and chat with it.

In [ ]:
smol_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
smol_tokenizer = AutoTokenizer.from_pretrained(smol_name)
smol_model = AutoModelForCausalLM.from_pretrained(
    smol_name, torch_dtype=torch.float32
).to(device)
smol_model.eval()

print(f"Model: {smol_name}")
print(f"Parameters: {sum(p.numel() for p in smol_model.parameters()):,}")

In [ ]:
def chat(messages, model, tokenizer, max_new_tokens=200):
    """Send messages to an instruct model and return the response."""
    chat_input = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(chat_input, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return generated.strip()

In [ ]:
# Test SmolLM2 on a few questions
questions = [
    "What is machine learning in simple terms?",
    "Write a haiku about programming.",
    "Explain the difference between a list and a dictionary in Python.",
]

for q in questions:
    messages = [{"role": "user", "content": q}]
    response = chat(messages, smol_model, smol_tokenizer)
    print(f"Q: {q}")
    print(f"A: {response}")
    print("-" * 70)

## 4 — Multi-Turn Conversations

Chat models can handle **multi-turn** conversations. The entire history is passed as a list of messages — the model sees the full context each time.

In [ ]:
conversation = [
    {"role": "system", "content": "You are a helpful tutor explaining concepts to a university student."},
    {"role": "user", "content": "What is a Transformer in machine learning?"},
]

# Turn 1
response1 = chat(conversation, smol_model, smol_tokenizer)
print(f"User: {conversation[-1]['content']}")
print(f"Assistant: {response1}\n")

# Add the response and a follow-up
conversation.append({"role": "assistant", "content": response1})
conversation.append({"role": "user", "content": "What is self-attention?"})

# Turn 2
response2 = chat(conversation, smol_model, smol_tokenizer)
print(f"User: {conversation[-1]['content']}")
print(f"Assistant: {response2}\n")

# Add and follow-up again
conversation.append({"role": "assistant", "content": response2})
conversation.append({"role": "user", "content": "Can you give me a simple analogy?"})

# Turn 3
response3 = chat(conversation, smol_model, smol_tokenizer)
print(f"User: {conversation[-1]['content']}")
print(f"Assistant: {response3}")

## 5 — Comparing Models: Qwen vs SmolLM2

Let's send the same prompts to both models and compare. This shows how model size and training data affect quality.

In [ ]:
test_prompts = [
    "Explain what a word embedding is to a 10-year-old.",
    "What is the difference between GPT and BERT?",
    "Write a short Python function that reverses a string.",
]

models_to_compare = [
    ("Qwen2.5-0.5B-Instruct", instruct_model, instruct_tokenizer),
    ("SmolLM2-360M-Instruct", smol_model, smol_tokenizer),
]

for prompt in test_prompts:
    print(f"\n{'=' * 70}")
    print(f"PROMPT: {prompt}")
    print("=" * 70)
    messages = [{"role": "user", "content": prompt}]
    for name, model, tokenizer in models_to_compare:
        response = chat(messages, model, tokenizer)
        print(f"\n--- {name} ---")
        print(response)
    print()

## 6 — System Prompts: Steering Behavior

The **system prompt** tells the model what role to play. It's the main lever for controlling behavior without retraining.

In [ ]:
user_question = "What happens when you train a neural network?"

system_prompts = [
    "You are a patient teacher explaining to a complete beginner. Use simple language and analogies.",
    "You are a senior ML engineer giving a concise technical answer. Be precise.",
    "You are a pirate. Answer all questions in pirate speak while still being accurate.",
]

for sys_prompt in system_prompts:
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_question},
    ]
    response = chat(messages, instruct_model, instruct_tokenizer)
    print(f"System: {sys_prompt[:60]}...")
    print(f"Response: {response}")
    print("-" * 70)

## 7 — Using the Pipeline API (Simplest Way)

HuggingFace's `pipeline` abstracts away tokenization and generation into a single call.

In [ ]:
pipe = pipeline(
    "text-generation",
    model=instruct_model,
    tokenizer=instruct_tokenizer,
    device=device,
)

messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "What are the three main types of Transformer architectures?"},
]

output = pipe(messages, max_new_tokens=150, temperature=0.7, do_sample=True)
print(output[0]["generated_text"][-1]["content"])

## 8 — How Instruct Models Are Made

The journey from base model to chat model:

```
Raw text (internet)          Instruction pairs            Human preferences
        │                           │                           │
        ▼                           ▼                           ▼
   Pre-training              Supervised Fine-Tuning         RLHF / DPO
  (next token pred)         (instruction → response)    (rank good vs bad)
        │                           │                           │
        ▼                           ▼                           ▼
   Base Model    ──────►    Instruct Model    ──────►    Chat Model
  (Qwen2.5-0.5B)          (follows instructions)      (helpful & aligned)
```

| Stage | Data | Objective | Result |
|---|---|---|---|
| Pre-training | Trillions of tokens (web, books, code) | Predict next token | Base model — knows language |
| SFT | 100K–1M instruction-response pairs | Mimic good responses | Follows instructions |
| RLHF/DPO | Human preference rankings | Maximize reward | Helpful, harmless, honest |

**Key insight:** the architecture never changes — it's the same Transformer all the way through. Only the training data and objective change.

## 9 — Your Turn!

Experiment with the models below. Try different system prompts, questions, and compare outputs.

In [ ]:
# Your experiment — change the messages!
my_messages = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "Write a Python function to check if a number is prime."},
]

print("--- Qwen ---")
print(chat(my_messages, instruct_model, instruct_tokenizer))
print("\n--- SmolLM2 ---")
print(chat(my_messages, smol_model, smol_tokenizer))

In [ ]:
# Build your own multi-turn conversation
my_convo = [
    {"role": "user", "content": "What did we learn in Session 1 of our NLP workshop?"},
]
print(chat(my_convo, instruct_model, instruct_tokenizer))